# 01 — EDA: Compensation Org-Frame

**Project H18 — Compensation Equity Analyzer.** 5,000 synthetic employees with role, level, dept, gender, nationality, age, tenure, education, monthly comp (AED), performance rating. A small unexplained gender gap is injected so the Blinder-Oaxaca decomposition has signal to recover.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
df = pd.read_parquet('../data/processed/org_frame.parquet')
df.head()

## 1. Shape, gender mix, role mix

In [ ]:
print(f'rows: {len(df):,}')
print(f'gender mix:\n{df["gender"].value_counts(normalize=True).round(3)}')
print(f'\nroles: {df["role"].nunique()}    depts: {df["dept"].nunique()}')
print('\nrole counts:')
print(df['role'].value_counts())

## 2. Compensation distribution overall and by gender

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df['monthly_comp_aed'], bins=40, color='#1f77b4', ax=axes[0])
axes[0].set_title('Monthly comp distribution (AED)')
sns.boxplot(data=df, x='gender', y='monthly_comp_aed', palette=['#1f77b4', '#fb6a4a'], ax=axes[1])
axes[1].set_title('Comp by gender')
plt.tight_layout(); plt.show()

## 3. Raw gap headline

In [ ]:
mu_m = df.loc[df['gender'] == 'M', 'monthly_comp_aed'].mean()
mu_f = df.loc[df['gender'] == 'F', 'monthly_comp_aed'].mean()
log_gap = np.log(mu_m) - np.log(mu_f)
print(f'mean comp M = {mu_m:,.0f}   F = {mu_f:,.0f}   raw log gap = {log_gap:+.3f}'
      f'   (~{(np.exp(log_gap) - 1) * 100:+.1f}% AED)')

## 4. Comp by role and gender

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5.5))
order = df.groupby('role')['monthly_comp_aed'].median().sort_values(ascending=False).index
sns.boxplot(data=df, y='role', x='monthly_comp_aed', hue='gender', order=order,
            palette=['#1f77b4', '#fb6a4a'], ax=ax)
ax.set_title('Compensation by role and gender')
plt.tight_layout(); plt.show()

## 5. Comp vs tenure scatter

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for g, c in [('M', '#1f77b4'), ('F', '#fb6a4a')]:
    sub = df[df['gender'] == g]
    ax.scatter(sub['tenure_yrs'], sub['monthly_comp_aed'], s=4, alpha=0.4, color=c, label=g)
ax.set_xlabel('tenure_yrs'); ax.set_ylabel('monthly comp (AED)')
ax.set_title('Comp vs tenure, by gender')
ax.legend(); plt.tight_layout(); plt.show()

## 6. Comp vs level

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.boxplot(data=df, x='level', y='monthly_comp_aed', hue='gender',
            palette=['#1f77b4', '#fb6a4a'], ax=ax)
ax.set_title('Comp by level and gender'); plt.tight_layout(); plt.show()

## 7. Performance rating × gender

In [ ]:
tab = pd.crosstab(df['performance_rating'], df['gender'], normalize='columns').round(3)
print(tab)
fig, ax = plt.subplots(figsize=(7, 4))
tab.plot(kind='bar', ax=ax, color=['#fb6a4a', '#1f77b4'])
ax.set_title('Performance-rating distribution by gender')
plt.tight_layout(); plt.show()

## 8. Composition decomposition — gender share by role

In [ ]:
comp = pd.crosstab(df['role'], df['gender'], normalize='index').round(3)
print(comp.sort_values('F', ascending=False))
fig, ax = plt.subplots(figsize=(11, 5))
comp.sort_values('F', ascending=False).plot(kind='barh', stacked=True,
                                              color=['#fb6a4a', '#1f77b4'], ax=ax)
ax.set_title('Gender share by role')
ax.set_xlabel('share'); plt.tight_layout(); plt.show()

## 9. Implications for the decomposition
- Raw log-comp gap is small but positive (M > F).
- Both groups span the full role / level distribution but the role-mix differs slightly — Blinder-Oaxaca will split the gap into a 'composition' (endowment) and a 'pay-for-the-same-thing' (coefficient/unexplained) part.
- The injected unexplained gap (~7 log-points) should show up as the C component.